# RSNA Knee Abnormality Detection: report audit and rule-based labeler (NB1)

Sprint 03 project notebook for the Kaggle competition *RSNA Knee Abnormality Detection*.

**Problem.** Predict 12 knee findings per MRI study (ACL, MCL, medial/lateral meniscus, medial/lateral/patellofemoral osteoarthritis, effusion, synovitis, Baker's cyst, contusion, fracture). Test studies contain images only. Training studies also include a free-text, multilingual radiology report, but only a small subset carries the 12 labels.

**Purpose of this notebook.** Understand the data, then build a rule-based labeler that turns reports into labels, and measure it against the studies that have real labels (the *gold* set).

**Contents**
1. Environment and data check
2. Label coverage, report languages, DICOM formats
3. Report audit and rule labeler v1
4. Rule labeler v2 (improvements after error analysis)

**Outputs saved:** `pseudo_labels_v1.csv`, `pseudo_labels_v2.csv`, and a `wheels/` folder with optional DICOM decoder packages.

## 1. Environment and data check
Checks the hardware, locates the competition files, summarises labels and series metadata, and tests that DICOM files decode.

In [1]:
import os, glob, time, random, subprocess, shutil, importlib
import numpy as np
import pandas as pd

### 1.1 Environment
GPU, memory, disk and available libraries.

In [2]:
try:
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "no nvidia-smi")
except Exception as e:
    print("nvidia-smi failed:", e)
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(),
          "| gpus:", torch.cuda.device_count())
except Exception as e:
    print("torch problem:", e)
try:
    import psutil
    print("RAM total GB: %.1f" % (psutil.virtual_memory().total / 1e9))
except Exception:
    pass
print("Disk /kaggle/working free GB: %.1f" % (shutil.disk_usage("/kaggle/working").free / 1e9))
for lib in ["pydicom", "pylibjpeg", "openjpeg", "gdcm", "cv2", "timm", "sklearn"]:
    try:
        m = importlib.import_module(lib)
        print(f"  {lib}: OK", getattr(m, "__version__", ""))
    except Exception:
        print(f"  {lib}: NOT installed")

GPU 0: Tesla T4 (UUID: GPU-de3102f6-717e-ee9a-93fb-e6b4369e2100)
GPU 1: Tesla T4 (UUID: GPU-14effaf4-7472-da24-65cd-07979192f028)
torch 2.10.0+cu128 | cuda available: True | gpus: 2
RAM total GB: 33.7
Disk /kaggle/working free GB: 20.9
  pydicom: OK 3.0.2
  pylibjpeg: NOT installed
  openjpeg: NOT installed
  gdcm: NOT installed
  cv2: OK 4.13.0
  timm: OK 1.0.26
  sklearn: OK 1.6.1


### 1.2 Data location
Finds the competition folder. A recursive search over about 820k files is slow, so candidate paths are checked directly.

In [3]:
print("/kaggle/input contains:", os.listdir("/kaggle/input"))
candidates = [
    "/kaggle/input/competitions/rsna-knee-abnormality-detection",
    "/kaggle/input/rsna-knee-abnormality-detection",
]
# also check one level down in every input folder (cheap, no deep walk)
for d in os.listdir("/kaggle/input"):
    candidates.append(f"/kaggle/input/{d}")
    if os.path.isdir(f"/kaggle/input/{d}"):
        for d2 in os.listdir(f"/kaggle/input/{d}"):
            candidates.append(f"/kaggle/input/{d}/{d2}")
ROOT = next((c for c in candidates if os.path.exists(f"{c}/train.csv")), None)
print("train.csv found in:", ROOT)
if ROOT is None:
    raise SystemExit("Competition data not attached. Use 'Add Input' and search for the competition.")
print("ROOT =", ROOT)
print("Files in ROOT:", sorted(os.listdir(ROOT)))

/kaggle/input contains: ['competitions']
train.csv found in: /kaggle/input/competitions/rsna-knee-abnormality-detection
ROOT = /kaggle/input/competitions/rsna-knee-abnormality-detection
Files in ROOT: ['sample_submission.csv', 'test.csv', 'test_series', 'test_series.csv', 'train.csv', 'train_series', 'train_series.csv']


### 1.3 Labels and reports
Size of the training table, label columns, number of labeled studies and a report sample.

In [4]:
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
train = pd.read_csv(f"{ROOT}/train.csv")
print("train shape:", train.shape)
print("columns:", list(train.columns))
present = [c for c in LABELS if c in train.columns]
print("label columns present:", len(present), "of 12")
labeled_mask = train[present].notna().all(axis=1)
print("studies with all 12 labels:", int(labeled_mask.sum()), "| without:", int((~labeled_mask).sum()))
if labeled_mask.sum():
    prev = train.loc[labeled_mask, present].mean().round(3)
    print("prevalence among labeled studies:\n", prev.to_string())
if "Report" in train.columns:
    rep = train["Report"].fillna("")
    print("reports empty:", int((rep.str.len() == 0).sum()),
          "| median length (chars):", int(rep.str.len().median()))
    print("sample report (first 300 chars):\n", rep.iloc[0][:300])

train shape: (4407, 14)
columns: ['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
label columns present: 12 of 12
studies with all 12 labels: 58 | without: 4349
prevalence among labeled studies:
 ACL                 0.414
MCL                 0.155
Medial Meniscus     0.448
Lateral Meniscus    0.397
Medial OA           0.259
Lateral OA          0.190
PF OA               0.362
Effusion            0.603
Synovitis           0.466
Baker's             0.207
Contusion           0.328
Fracture            0.310
reports empty: 0 | median length (chars): 977
sample report (first 300 chars):
 Técnica: RMN de la rodilla. Resultados: Rotura de menisco interno. Signo de necrosis avascular subcondral en el cóndilo femoral medial. Artrosis femorotibial medial. Derrame. . Impresión: Rotura de menisco interno. Signo de necrosis avascular subcondral en el cóndilo femoral medi

### 1.4 Series metadata
Imaging planes, series per study, and the relation between the fluid-sensitive and fat-suppression flags.

In [5]:
ts = pd.read_csv(f"{ROOT}/train_series.csv")
print("train_series shape:", ts.shape, "| studies:", ts.StudyInstanceUID.nunique())
print(ts.Anatomical_Plane.value_counts().to_string())
print("series per study (describe):\n", ts.groupby("StudyInstanceUID").size().describe().round(1).to_string())
print("Fluid_Sensitive x Fat_Suppression:\n", pd.crosstab(ts.Fluid_Sensitive, ts.Fat_Suppression).to_string())
test = pd.read_csv(f"{ROOT}/test.csv")
print("example test.csv rows:", len(test), "(real test has ~1300 studies)")

train_series shape: (24371, 5) | studies: 4407
Anatomical_Plane
Sagittal    9864
Coronal     8609
Axial       5898
series per study (describe):
 count    4407.0
mean        5.5
std         1.4
min         3.0
25%         5.0
50%         5.0
75%         6.0
max        14.0
Fluid_Sensitive x Fat_Suppression:
 Fat_Suppression      0      1
Fluid_Sensitive              
0                10361      0
1                    0  14010
example test.csv rows: 3 (real test has ~1300 studies)


### 1.5 DICOM decode test
Reads a sample of slices to check transfer syntaxes and decoding speed.

In [6]:
import pydicom
random.seed(0)
sample_series = ts.sample(min(60, len(ts)), random_state=0)
by_syntax, failures, times = {}, [], []
for _, r in sample_series.iterrows():
    folder = f"{ROOT}/train_series/{r.StudyInstanceUID}/{r.SeriesInstanceUID}"
    files = sorted(glob.glob(folder + "/*.dcm"))
    if not files:
        continue
    f = files[len(files) // 2]
    try:
        hdr = pydicom.dcmread(f, stop_before_pixels=True)
        syn = str(hdr.file_meta.TransferSyntaxUID.name)
    except Exception as e:
        failures.append((f, "header: " + str(e)[:80])); continue
    by_syntax.setdefault(syn, [0, 0])
    by_syntax[syn][0] += 1
    t0 = time.time()
    try:
        arr = pydicom.dcmread(f).pixel_array
        times.append(time.time() - t0)
        by_syntax[syn][1] += 1
    except Exception as e:
        failures.append((syn, str(e)[:100]))
print("transfer syntax -> [tested, decoded OK]:")
for k, v in by_syntax.items():
    print("  ", k, v)
print("decode failures:", len(failures))
for fl in failures[:5]:
    print("  ", fl)
if times:
    print("avg single-slice read+decode: %.1f ms" % (1000 * np.mean(times)))

transfer syntax -> [tested, decoded OK]:
   Explicit VR Little Endian [60, 60]
decode failures: 0
avg single-slice read+decode: 9.0 ms


### 1.6 Series read speed
Time to read one complete series.

In [7]:
r = ts.iloc[0]
folder = f"{ROOT}/train_series/{r.StudyInstanceUID}/{r.SeriesInstanceUID}"
files = sorted(glob.glob(folder + "/*.dcm"))
t0 = time.time()
shapes = []
for f in files:
    try:
        shapes.append(pydicom.dcmread(f).pixel_array.shape)
    except Exception as e:
        shapes.append("ERR")
print(f"{len(files)} slices in {time.time() - t0:.2f}s | unique shapes: {set(map(str, shapes))}")
print("Checks complete.")

22 slices in 0.55s | unique shapes: {'(512, 512)'}
Checks complete.


**Findings**
- 4,407 training studies and 24,371 series (about 5.5 per study, three planes).
- Only **58** studies have all 12 labels, so the reports must supply labels for the rest.
- All sampled DICOM files use the same transfer syntax and decode without errors; the example series read has 512x512 slices.
- `Fluid_Sensitive` and `Fat_Suppression` are identical in the training series.

## 2. Label coverage, report languages and DICOM formats
A second round of checks: which labels exist, which languages the reports use, whether any DICOM transfer syntaxes other than the standard one occur, and whether extra decoders are needed.

In [8]:
import os, sys, time, subprocess
import pandas as pd
import pydicom
from concurrent.futures import ThreadPoolExecutor

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
print("CPU cores:", os.cpu_count())

CPU cores: 4


### 2.1 Label coverage
Number of non-null labels per column and the values present.

In [9]:
train = pd.read_csv(f"{ROOT}/train.csv")
print("non-null count per label:\n", train[LABELS].notna().sum().to_string())
print("studies with at least one label:", int(train[LABELS].notna().any(axis=1).sum()))
print("values seen in label columns:", sorted(train[LABELS].stack().unique().tolist()))

non-null count per label:
 ACL                 58
MCL                 58
Medial Meniscus     58
Lateral Meniscus    58
Medial OA           58
Lateral OA          58
PF OA               58
Effusion            58
Synovitis           58
Baker's             58
Contusion           58
Fracture            58
studies with at least one label: 58
values seen in label columns: [0.0, 1.0]


### 2.2 Report languages
Automatic language detection (`langdetect`). Counts are approximate.

In [10]:
try:
    from langdetect import detect, DetectorFactory
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langdetect"])
    from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def lang(t):
    try:
        return detect(str(t)[:500])
    except Exception:
        return "unk"

train["lang"] = train.Report.map(lang)
print(train.lang.value_counts().head(15).to_string())
labeled = train[LABELS].notna().all(axis=1)
print("languages among the fully labeled studies:\n", train.loc[labeled, "lang"].value_counts().to_string())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.3 MB/s eta 0:00:00
lang
en    1736
es     682
tr     546
hr     406
el     321
de     262
bg     220
nl     153
fr      81
languages among the fully labeled studies:
 lang
en    28
es    10
tr     6
hr     4
bg     3
el     3
nl     2
de     2


### 2.3 Optional DICOM decoders
Installs `pylibjpeg` packages and keeps the wheel files for offline use.

In [11]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"],
                   capture_output=True, text=True)
print("pip install return code:", r.returncode)
if r.returncode != 0:
    print(r.stderr[-300:])
# keep wheels for the offline submission notebook later
r2 = subprocess.run([sys.executable, "-m", "pip", "download", "-q",
                     "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "-d", "/kaggle/working/wheels"],
                    capture_output=True, text=True)
print("wheel download return code:", r2.returncode)

pip install return code: 0
wheel download return code: 0


### 2.4 Header scan
Reads one header per series for about 6,000 series: transfer syntax, slices per series and image size.

In [12]:
ts = pd.read_csv(f"{ROOT}/train_series.csv")
sub = ts.sample(min(6000, len(ts)), random_state=1)

def probe(row):
    folder = f"{ROOT}/train_series/{row.StudyInstanceUID}/{row.SeriesInstanceUID}"
    try:
        fs = sorted(os.listdir(folder))
        path = f"{folder}/{fs[0]}"
        h = pydicom.dcmread(path, stop_before_pixels=True)
        return dict(n=len(fs), syntax=str(h.file_meta.TransferSyntaxUID.name),
                    size=f"{int(h.Rows)}x{int(h.Columns)}", path=path)
    except Exception as e:
        return dict(n=0, syntax="ERR " + str(e)[:60], size="?", path=folder)

t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    res = pd.DataFrame(list(ex.map(probe, list(sub.itertuples(index=False)))))
print("scan took %.0fs" % (time.time() - t0))
print("transfer syntaxes:\n", res.syntax.value_counts().to_string())
print("slices per series:\n", res.n.describe().round(1).to_string())
print("top image sizes:\n", res["size"].value_counts().head(8).to_string())

scan took 14s
transfer syntaxes:
 syntax
Explicit VR Little Endian    6000
slices per series:
 count    6000.0
mean       33.6
std        27.1
min        11.0
25%        25.0
50%        30.0
75%        33.0
max       320.0
top image sizes:
 size
512x512      1449
640x640       812
384x384       700
320x320       432
1024x1024     276
256x256       239
704x704       163
480x480       158


### 2.5 Non-standard syntaxes
Attempts to decode an example of any syntax other than Explicit VR Little Endian.

In [13]:
for syn in res.syntax.unique():
    if syn == "Explicit VR Little Endian" or syn.startswith("ERR"):
        continue
    p = res[res.syntax == syn].path.iloc[0]
    try:
        a = pydicom.dcmread(p).pixel_array
        print("OK  ", syn, a.shape, a.dtype)
    except Exception as e:
        print("FAIL", syn, str(e)[:150])
print("Checks complete.")

Checks complete.


**Findings**
- All 58 labeled studies have every one of the 12 labels; there are no partially labeled studies.
- Reports span at least nine detected languages (English 1,736; Spanish 682; Turkish 546; Croatian 406; Greek 321; German 262; Bulgarian 220; Dutch 153; French 81).
- All scanned series use Explicit VR Little Endian; slices per series have a median of 30 and image sizes range from 256 to 1024 pixels.

## 3. Report audit and rule labeler v1
Builds a first keyword-based labeler for English, Spanish, German and Turkish reports and evaluates it on the 58 gold studies.

### 3.1 Setup
Loads the training table and marks the gold (fully labeled) studies.

In [14]:
import re, sys, subprocess
import numpy as np, pandas as pd
ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
train = pd.read_csv(f"{ROOT}/train.csv")
gold_mask = train[LABELS].notna().all(axis=1)
print("studies:", len(train), "| gold (labeled):", int(gold_mask.sum()))

studies: 4407 | gold (labeled): 58


### 3.2 Audit: report length and language
Language detection, report length and a duplicate check.

In [15]:
try:
    from langdetect import detect, DetectorFactory
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langdetect"])
    from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
def lang(t):
    try: return detect(str(t)[:500])
    except Exception: return "unk"
train["lang"] = train.Report.map(lang)
train["n_chars"] = train.Report.str.len()
print(train.lang.value_counts().to_string())
print(train.n_chars.describe().round(0).to_string())
print("duplicate StudyInstanceUID:", int(train.StudyInstanceUID.duplicated().sum()))

lang
en    1736
es     682
tr     546
hr     406
el     321
de     262
bg     220
nl     153
fr      81
count    4407.0
mean     1098.0
std       694.0
min        52.0
25%       588.0
50%       977.0
75%      1460.0
max      4743.0
duplicate StudyInstanceUID: 0


### 3.3 Manual inspection of reports
Three reports per language next to the gold labels (when available). Reading examples by hand guides the rule design.

In [16]:
# 3 reports per language, shown with gold labels when available
for lg in ["en", "es", "tr", "hr", "el"]:
    print("=" * 70, lg)
    for _, r in train[train.lang == lg].head(3).iterrows():
        pos = [l for l in LABELS if r[l] == 1] if r[LABELS].notna().all() else "(no gold)"
        print(r.Report[:500].replace("\n", " "), "\n   -> gold positives:", pos, "\n")

====================================================================== en
 In the medial compartment, the meniscus is not torn and there is no focal chondrosis or chondral injury.   The lateral compartment, there is suspicious incomplete discoid lateral meniscus without tear. There is no focal chondrosis or chondral injury.   The patellofemoral joint, the patellar and trochlear cartilage appears congruent without chondral lesion. The medial patellofemoral ligament is intact.    In the intercondylar notch, the anterior and posterior cruciate ligaments are intact. There 
   -> gold positives: (no gold) 

MRI of left knee with -Locator, SG PD FatSat, SG T2W FS, SG T1W, CO T2W FS, AX T2W FS  Findings: > The contour of anterior cruciate ligament is in a straight line but increase signal intensity inside, likely due to sprain > One cystic lesion posterior to ACL, maybe due to ganglion cyst > The contour of posterior cruciate ligaments appearing within normal limit > No definite tear of media

### 3.4 Rule-based labeler v1
Per finding: keyword patterns in English, Spanish, German and Turkish, a negation check, and clause splitting. Ligament and meniscus findings also require a damage word in the same clause.

In [17]:
NEG = re.compile(r"\b(no|not|without|absent|intact|normal|negative|sin|ausencia|keine?n?|ohne|unauff\w+|yok)\b"
                 r"|izlenmem|saptanmam|g\u00f6r\u00fclmem|g\u00f6zlenmem|bulunmam", re.I)
SPLIT = re.compile(r"[.;:\n]| but | however | pero | sin embargo | aber | ancak | fakat ", re.I)
DAMAGE = re.compile(r"tear|torn|rupture|ruptur|partial|sprain|injur|lesion|rotura|ruptura|lesi\u00f3n|"
                    r"riss|verletz|y\u0131rt|r\u00fcpt\u00fcr|kopma|zedelen|hasar", re.I)
OA = re.compile(r"osteoarth|arthrosis|artrosis|arthrose|gonarthrose|gonartroz|osteoartrit|"
                r"cartilage (loss|thinning)|chondromalacia|condromalacia|degenerat", re.I)
MENISCUS = re.compile(r"menisc|menisk", re.I)
MEDIAL = re.compile(r"medial|interno|mediale|\bi\u00e7\b", re.I)
LATERAL = re.compile(r"lateral|externo|laterale|d\u0131\u015f", re.I)

STRUCT = {  # structure patterns that also need a DAMAGE word in the same clause
    "ACL": r"anterior cruciate|\bacl\b|cruzado anterior|\blca\b|vordere[sn]? kreuzband|\u00f6n \u00e7apraz",
    "MCL": r"medial collateral|\bmcl\b|colateral (medial|tibial)|ligamento lateral interno|innenband|"
           r"mediale[sn]? kollateral|i\u00e7 yan ba\u011f",
    "Medial Meniscus": r"medial menisc|menisco (interno|medial)|innenmeniskus|mediale[rn]? meniskus|i\u00e7 menisk|medial menisk",
    "Lateral Meniscus": r"lateral menisc|menisco (externo|lateral)|au(\u00df|ss)enmeniskus|laterale[rn]? meniskus|d\u0131\u015f menisk|lateral menisk",
}
PRESENCE = {  # presence of the word alone is enough
    "PF OA_site": r"patell?o?femor|femoro-?patel|femoropatel|retropatel|rotuliana|patelofemoral",
    "Effusion": r"effusion|derrame|erguss|ef\u00fczyon|eklem s\u0131v\u0131|joint fluid",
    "Synovitis": r"synovitis|sinovitis|sinovit|synovialitis|synovial (thickening|inflamm)",
    "Baker's": r"baker|popliteal cyst|quiste popl\u00edteo|kniekehlenzyste",
    "Contusion": r"contusion|contusi\u00f3n|kontusion|kont\u00fczyon|bone (marrow )?(oedema|edema|bruis)|edema \u00f3seo|"
                 r"knochenmark(\u00f6|oe)dem|kemik ili\u011fi \u00f6dem|kemik \u00f6dem",
    "Fracture": r"fracture|fractura|fraktur|k\u0131r\u0131k",
}
STRUCT = {k: re.compile(v, re.I) for k, v in STRUCT.items()}
PRESENCE = {k: re.compile(v, re.I) for k, v in PRESENCE.items()}

def label_report(text):
    out = {l: 0 for l in LABELS}
    for clause in SPLIT.split(str(text)):
        c = clause.strip()
        if not c or NEG.search(c):
            continue
        for k, rx in STRUCT.items():
            if rx.search(c) and DAMAGE.search(c):
                out[k] = 1
        for k, rx in PRESENCE.items():
            if rx.search(c):
                if k == "PF OA_site":
                    if OA.search(c): out["PF OA"] = 1
                else:
                    out[k] = 1
        if OA.search(c) and not MENISCUS.search(c):
            if MEDIAL.search(c): out["Medial OA"] = 1
            if LATERAL.search(c): out["Lateral OA"] = 1
    return out

pred = pd.DataFrame([label_report(t) for t in train.Report], index=train.index)
print(pred.mean().round(3).to_string())   # predicted prevalence over ALL studies

ACL                 0.114
MCL                 0.063
Medial Meniscus     0.249
Lateral Meniscus    0.111
Medial OA           0.082
Lateral OA          0.063
PF OA               0.057
Effusion            0.338
Synovitis           0.099
Baker's             0.229
Contusion           0.164
Fracture            0.053


### 3.5 Evaluation against the 58 gold studies
Precision, recall and F1 per finding.

In [18]:
from sklearn.metrics import precision_recall_fscore_support
g = train[gold_mask]
rows = []
for l in LABELS:
    p, r, f, _ = precision_recall_fscore_support(g[l].astype(int), pred.loc[g.index, l], average="binary", zero_division=0)
    rows.append(dict(finding=l, precision=round(p, 2), recall=round(r, 2), f1=round(f, 2),
                     gold_pos=int(g[l].sum()), pred_pos=int(pred.loc[g.index, l].sum())))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("mean F1:", res.f1.mean().round(3))

         finding  precision  recall   f1  gold_pos  pred_pos
             ACL       0.80    0.67 0.73        24        20
             MCL       0.56    0.56 0.56         9         9
 Medial Meniscus       0.81    0.65 0.72        26        21
Lateral Meniscus       0.83    0.65 0.73        23        18
       Medial OA       0.83    0.33 0.48        15         6
      Lateral OA       0.50    0.18 0.27        11         4
           PF OA       0.50    0.05 0.09        21         2
        Effusion       0.65    0.63 0.64        35        34
       Synovitis       0.73    0.41 0.52        27        15
         Baker's       0.57    0.67 0.62        12        14
       Contusion       0.48    0.63 0.55        19        25
        Fracture       0.78    0.39 0.52        18         9
mean F1: 0.536


### 3.6 Error inspection
Reports where the labeler and the gold labels disagree.

In [19]:
for l in ["ACL", "Medial Meniscus", "Effusion"]:   # change the finding to inspect others
    wrong = g[g[l].astype(int) != pred.loc[g.index, l]]
    print("=" * 70, l, "| errors:", len(wrong))
    for _, r in wrong.head(4).iterrows():
        print(f"[{r.lang}] gold={int(r[l])} pred={int(pred.loc[r.name, l])} :: {r.Report[:350]!r}\n")

====================================================================== ACL | errors: 12
[en] gold=0 pred=1 :: ' In the medial compartment, the meniscus is not torn and there is no focal chondrosis or chondral injury. \n  \n The lateral compartment, there is complex tear (main radial and longitudinal vertical component) at the posterior horn of the lateral meniscus. There is no focal chondrosis or chondral injury. \n  \n The patellofemoral joint, the patellar an'

[en] gold=0 pred=1 :: 'MRI of Knee with \n-Locator, SG PD FatSat, SG T2W FS, SG T1W, CO T2W FS, AX T2W FS\n\n\n\n* High signal in the soft tissue medial to the medial collateral ligament with mild increased signal intensity of proximal ligament, suspect grade 2 injury.\n* Interstitial hyperintense signal of ACL, in favor of grade 1 injury.\n* Bone marrow edema in bilateral femo'

[en] gold=0 pred=1 :: '> MRI of knee was performed in sagittal, axial and coronal sections.\n> Spin echo sequence were used.\n\n> Split fractures at 

### 3.7 Save v1 pseudo-labels
Saved for reference; superseded by v2.

In [20]:
out = train[["StudyInstanceUID", "lang"]].join(pred)
out.to_csv("/kaggle/working/pseudo_labels_v1.csv", index=False)
print("saved", out.shape)

saved (4407, 14)


**Findings (v1)**
- Mean F1 on the 58 gold studies: **0.536**. Weakest findings: PF OA (recall 0.05), Lateral OA, Medial OA, Fracture and Synovitis.
- Rules only cover English, Spanish, German and Turkish (3,226 of 4,407 studies). Studies in other languages received all-zero labels, which are false negatives.
- Cartilage and osteoarthritis findings are rarely written with the word "osteoarthritis"; they appear as cartilage loss or chondral defects in a compartment.
- Grade 1 sprains and interstitial signal changes were counted as tears, but the gold labels treat them as negative.
- Gold labels can disagree with the report text, so reports are a noisy proxy for the labels.

## 4. Rule labeler v2
Improvements from the v1 error analysis: Unicode normalisation, ligament and meniscus findings require a real tear, cartilage and compartment rules for osteoarthritis (with compartment context carried across clauses), and studies in uncovered languages are left unlabeled instead of labeled negative.

### 4.1 Labeler v2
Defines the normalisation, the patterns, the negation logic and a `why()` helper that shows which clause triggered a label.

In [21]:
import re, unicodedata
import numpy as np, pandas as pd

COVERED = {"en", "es", "de", "tr"}      # languages the rules were written for

def norm(t):
    t = unicodedata.normalize("NFKC", str(t))      # fixes the Greek micro sign and similar lookalikes
    t = t.replace("\u0130", "i")                   # Turkish dotted capital I
    t = t.lower().replace("\u0307", "")
    return t

NEG = re.compile(r"\b(no|not|without|absent|intact|normal|negative|unremarkable|sin|ausencia|keine?n?|ohne|unauff\w+|yok)\b"
                 r"|izlenmem|saptanmam|g\u00f6r\u00fclmem|g\u00f6zlenmem|bulunmam|normaldir|do\u011fal|"
                 r"no hay|no se observa|sin signos|conservad|l\u00edmites normales", re.I)
SPLIT = re.compile(r"[.;:\n>*]| but | however | pero | sin embargo | aber | ancak | fakat |, and (?=the )", re.I)

TEAR = re.compile(r"\btear|\btorn|rupture|ruptur|avulsion|complete disruption|rotura|ruptura|desgarro|\briss|"
                  r"y\u0131rt|r\u00fcpt\u00fcr|kopma|kopuk|\bk\u0131r\u0131k\w* (bant|ba\u011f)", re.I)
SOFT = re.compile(r"sprain|grade (1|i)\b|interstitial|mucoid|increased signal|signal (change|alteration)|"
                  r"distensi\u00f3n|esguince|zerrung|sinyal art", re.I)
LIG_INJ = re.compile(r"sprain|injur|grade (2|3|ii|iii)\b|esguince|lesi\u00f3n|verletz|zerrung|zedelen|hasar|\bpartial", re.I)

ACL = re.compile(r"anterior cruciate|\bacl\b|cruzado anterior|\blca\b|vordere[sn]? kreuzband|\bvkb\b|\u00f6n \u00e7apraz", re.I)
MCL = re.compile(r"medial collateral|\bmcl\b|colateral (medial|tibial)|\blcm\b|innenband|mediale[sn]? kollateral|"
                 r"i\u00e7 yan ba\u011f|medial kollateral", re.I)
MENISC = re.compile(r"menisc|menisk|men\u00edsc", re.I)
MEDIAL = re.compile(r"medial|interno|mediale|innen|\bi\u00e7\b|medyal|medij", re.I)
LATERAL = re.compile(r"lateral|externo|laterale|au(\u00df|ss)en|d\u0131\u015f|lateral", re.I)
PATELLA = re.compile(r"patell|trochle|rotul|patelof|femoropatel|retropatel|troclea|trochlea|rotula|patella|femoropatellar|"
                     r"femoro-patellar|patelofemoral|patellofemoral", re.I)
CART = re.compile(r"cartilag|chondral|chondr|cart\u00edlago|condral|condr|knorpel|k\u0131k\u0131rdak|hrskavic", re.I)
CDEF = re.compile(r"loss|thinning|defect|lesion|fissur|ulcer|erosion|denud|delaminat|full[- ]thickness|partial[- ]thickness|"
                  r"grade (2|3|4|ii|iii|iv)\b|chondromalacia|chondrosis|degenerat|malacia|\u00falcera|defecto|p\u00e9rdida|"
                  r"adelgaz|fisura|erosi\u00f3n|verlust|defekt|ausd\u00fcnn|kay\u0131p|kayb|incel|dejener|stanjen|lezij|osteophyt|osteofit", re.I)
OAW = re.compile(r"osteoarth|arthrosis|artrosis|arthrose|gonarthrose|gonartroz|osteoartrit|gonartroz|femorotibial osteo", re.I)
COMPT = re.compile(r"compartment|compartimento|kompartiment|kompartment|b\u00f6l\u00fcm|plateau|condyle|c\u00f3ndilo|kondyl|kondil|"
                   r"femorotibial|tibiofemoral|tibial", re.I)

PRES = {
    "Effusion": re.compile(r"effusion|derrame|erguss|ef\u00fczyon|s\u0131v\u0131 miktar\u0131 (hafif )?art|eklem s\u0131v\u0131s\u0131 art|hydrops", re.I),
    "Synovitis": re.compile(r"synovitis|sinovitis|sinovit|synovialitis|synovial (thickening|inflamm|proliferat|hypertroph|reaction)|"
                            r"s\u0131nov\u0131yal (kal\u0131nla|inflam)|pigmented villonodular", re.I),
    "Baker's": re.compile(r"baker|popliteal cyst|poplite\w* cyst|quiste popl\u00edteo|quistes popl|kniekehlenzyste|poplitea[l]? kist", re.I),
    "Contusion": re.compile(r"contusion|contusi\u00f3n|kontusion|kont\u00fczyon|bone (marrow )?(oedema|edema|bruis)|edema \u00f3seo|"
                            r"knochenmark(\u00f6|oe)dem|kemik ili\u011fi \u00f6dem|kemik \u00f6dem|bone bruise|m\u00e9dula \u00f3sea.*edema", re.I),
    "Fracture": re.compile(r"fracture|fractura|fraktur|k\u0131r\u0131\u011f\u0131|k\u0131r\u0131k", re.I),
}

def label_report(text, trace=False):
    """Returns dict label->0/1; with trace=True also dict label->triggering clause."""
    out = {l: 0 for l in LABELS}
    why = {}
    ctx = None                                           # last compartment header seen: medial / lateral / pf
    def setl(k, clause):
        out[k] = 1
        why.setdefault(k, clause[:160])
    for clause in SPLIT.split(norm(text)):
        c = clause.strip()
        if not c:
            continue
        has_side_m, has_side_l, has_pf = MEDIAL.search(c), LATERAL.search(c), PATELLA.search(c)
        if COMPT.search(c) and not CART.search(c) and not TEAR.search(c):      # a header-like clause: remember it
            ctx = "pf" if has_pf else ("medial" if has_side_m else ("lateral" if has_side_l else ctx))
        if NEG.search(c):
            continue
        # ligaments and menisci need a real tear (grade 1 / sprain stays negative)
        if TEAR.search(c) or not SOFT.search(c):
            if ACL.search(c) and TEAR.search(c): setl("ACL", c)
            if MENISC.search(c) and TEAR.search(c):
                if has_side_m: setl("Medial Meniscus", c)
                if has_side_l: setl("Lateral Meniscus", c)
        if MCL.search(c) and (TEAR.search(c) or LIG_INJ.search(c)): setl("MCL", c)
        # cartilage / osteoarthritis per compartment
        cart_hit = (CART.search(c) and CDEF.search(c)) or OAW.search(c)
        if cart_hit and not (MENISC.search(c) and not CART.search(c)):
            side = "pf" if has_pf else ("medial" if has_side_m else ("lateral" if has_side_l else ctx))
            if side == "pf": setl("PF OA", c)
            elif side == "medial": setl("Medial OA", c)
            elif side == "lateral": setl("Lateral OA", c)
        for k, rx in PRES.items():
            if rx.search(c): setl(k, c)
    return (out, why) if trace else out

def why(row_idx, label):
    """Debug helper: show which clause fired a label (or that none did) next to the gold value."""
    r = train.loc[row_idx]
    o, w = label_report(r.Report, trace=True)
    print(f"[{r.lang}] gold={r.get(label)} pred={o[label]}\n  trigger: {w.get(label, '(none)')}\n  report : {str(r.Report)[:600]!r}")

### 4.2 Run, mask uncovered languages and evaluate
Evaluates on gold studies in covered languages. The gold set is split into two halves as an overfitting check: tune on half A, verify on half B.

In [22]:
from sklearn.metrics import precision_recall_fscore_support
pred = pd.DataFrame([label_report(t) for t in train.Report], index=train.index)
covered = train.lang.isin(COVERED)
pred.loc[~covered, :] = np.nan                      # unlabeled, NOT negative
print("covered studies:", int(covered.sum()), "| left unlabeled:", int((~covered).sum()))
print("predicted prevalence (covered only):\n", pred[covered].mean().round(3).to_string())

def evaluate(g, name):
    rows = []
    for l in LABELS:
        y, p = g[l].astype(int), pred.loc[g.index, l].astype(int)
        pr, rc, f, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
        rows.append(dict(finding=l, precision=round(pr, 2), recall=round(rc, 2), f1=round(f, 2),
                         gold_pos=int(y.sum()), pred_pos=int(p.sum())))
    res = pd.DataFrame(rows); print(f"--- {name} (n={len(g)}) mean F1 = {res.f1.mean():.3f}"); return res

g_all = train[gold_mask]
g_cov = g_all[g_all.lang.isin(COVERED)]
print(evaluate(g_cov, "gold, covered languages").to_string(index=False))
# overfitting check: tune on one half, look at the other half
evaluate(g_cov.iloc[::2], "half A (tune here)"); evaluate(g_cov.iloc[1::2], "half B (check here)")
print("gold studies in uncovered languages:", int((~g_all.lang.isin(COVERED)).sum()), "-> evaluate these later with the transformer")

covered studies: 3226 | left unlabeled: 1181
predicted prevalence (covered only):
 ACL                 0.132
MCL                 0.119
Medial Meniscus     0.357
Lateral Meniscus    0.157
Medial OA           0.184
Lateral OA          0.126
PF OA               0.252
Effusion            0.451
Synovitis           0.140
Baker's             0.199
Contusion           0.209
Fracture            0.057
--- gold, covered languages (n=46) mean F1 = 0.696
         finding  precision  recall   f1  gold_pos  pred_pos
             ACL       0.89    0.94 0.92        18        19
             MCL       0.50    1.00 0.67         7        14
 Medial Meniscus       0.81    0.89 0.85        19        21
Lateral Meniscus       0.89    0.89 0.89        18        18
       Medial OA       0.83    0.56 0.67         9         6
      Lateral OA       0.60    0.67 0.63         9        10
           PF OA       0.50    0.38 0.43        16        12
        Effusion       0.65    0.85 0.73        26        34
     

### 4.3 Debug examples
Effusion false positives, with the triggering clause.

In [23]:
wrong = g_cov[g_cov["Effusion"].astype(int) != pred.loc[g_cov.index, "Effusion"].astype(int)]
for idx in wrong.index[:4]:
    why(idx, "Effusion")

[en] gold=0.0 pred=1
  trigger: moderate joint effusion, distended suprapatellar bursa with thickend synovial tissue are observed
  report : ' The study reveals normal knee joint alignment.   No fracture is seen.  \n   ACL is intact.   PCL is preserved.  \n  The MCL is intact.    Medial meniscus is not torn.  \n  The FCL, popliteus and biceps femoris tendons are preserved.       \n   Horizontal tear at anterior horn of the lateral meniscus is noted.  \n   Focal osteochondral defect at lateral patellar facet, about 8x12mm. with subchondral bone edema is detected.   Full thickness cartilage defect, 1.2x1.5cm. at lateral trochlea with subchondral bone edema is noted.   Small osteochondral body, about intact9xintact4cm. at anterior infrapate'
[en] gold=0.0 pred=1
  trigger: small joint effusion
  report : 'Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Date and Time: [DATE] [TIME]\nIndication: 2 years of pain with walking up and down the stairs twisting and squatting.\nComparison: No relevant

### 4.4 Save v2 pseudo-labels
NaN marks studies in languages the rules do not cover.

In [24]:
out = train[["StudyInstanceUID", "lang"]].join(pred)
out["covered"] = covered.values
out.to_csv("/kaggle/working/pseudo_labels_v2.csv", index=False)
print("saved", out.shape)

saved (4407, 15)


**Findings (v2)**
- Mean F1 on the 46 gold studies in covered languages: **0.696** (half A 0.723, half B 0.652), up from 0.536 for v1.
- Best findings: ACL (0.92), lateral meniscus (0.89), medial meniscus (0.85). Weakest: PF OA (0.43).
- In the four effusion false positives inspected, the reports describe small or moderate effusions that the gold labels count as negative.
- 1,181 studies (Croatian, Greek, Bulgarian, Dutch, French) remain unlabeled by rules and are handled by the text models in NB2.